# Clase 3 — Métodos Monte Carlo y Diferencias Temporales TD(0) (versión R)

**Curso:** Aprendizaje por Refuerzo: Fundamentos y Aplicaciones
**Institución:** Universidad Austral — Facultad de Ingeniería (Posgrados)
**Docente:** Dr. Darío Ezequiel Díaz
**Fecha:** 19 de mayo de 2026

---

## Resumen del cuaderno

Esta es la contraparte en **R** del cuaderno Python `Clase03_MC_TD0.ipynb`. Reproduce idiomáticamente los siete bloques del original empleando:

- **`R6`** para clases orientadas a objetos (entornos).
- **`ggplot2`** para visualizaciones de calidad de publicación (con `facet_wrap` para composiciones).
- **`ReinforcementLearning`** (sección complementaria) como contraste con una implementación tabular ya disponible en el ecosistema R.
- **`reticulate`** opcionalmente, como puente con Python para corroborar resultados frente al cuaderno principal.

La estructura sigue, por orden, los mismos bloques temáticos que el cuaderno Python:

1. **Entornos de trabajo**: GridWorld estocástico y Random Walk de 5 estados, definidos como clases R6.
2. **Métodos Monte Carlo**: implementación de `first_visit_mc_prediction` y `every_visit_mc_prediction` en R puro.
3. **Diferencias Temporales TD(0)**: implementación de `td0_prediction` con paso constante.
4. **Réplica de la figura 6.2** de Sutton & Barto: convergencia comparada sobre el random walk.
5. **Análisis de sensibilidad al paso $\alpha$**.
6. **Análisis de varianza** con $N = 500$ réplicas y bandas de confianza al 95%.
7. **GridWorld**: comparación entre la solución exacta y los estimadores MC y TD(0).

Una sección final ilustra el uso del paquete `ReinforcementLearning` como herramienta canónica del ecosistema R para tareas tabulares.

> **Compatibilidad.** Se ejecuta en R 4.3 o superior, tanto en entorno local (Windows/Linux/macOS) como en Google Colab con el kernel `ir` instalado. En el entorno local del docente, la combinación recomendada es **R 4.5.2 + Rtools 4.4 + Miniconda** con el entorno conda `rl-docencia`. Si se desea contrastar con Python, basta cargar `reticulate` y apuntar al entorno conda.

---

### Notación recurrente

| Símbolo | Significado |
|---------|-------------|
| $V^\pi(s)$ | Función de valor de estado bajo la política $\pi$ |
| $G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$ | Retorno descontado desde el paso $t$ |
| $\widehat{V}(s)$ | Estimador empírico de $V^\pi(s)$ |
| $\delta_t = R_{t+1} + \gamma \widehat{V}(S_{t+1}) - \widehat{V}(S_t)$ | TD-error en el paso $t$ |
| $\alpha$ | Paso de aprendizaje |
| $\gamma$ | Factor de descuento, $\gamma \in [0, 1)$ |

## Bloque 0 — Configuración del entorno

Importamos los paquetes R necesarios y fijamos la semilla global. Si dispone de un script personal de configuración en Drive (por ejemplo, `setup_R_colab.R`), este se carga automáticamente; en caso contrario, se instalan los paquetes mínimos para esta clase.

In [ ]:
# --- Paso 1: paquetes R ---
ruta_setup_personal <- "/content/drive/MyDrive/R_Colab/setup_R_colab.R"
if (file.exists(ruta_setup_personal)) {
  cat("Cargando setup personal desde Drive...\n")
  source(ruta_setup_personal)
} else {
  cat("Setup personal no encontrado. Instalando paquetes minimos para Clase 3...\n")
  paquetes_clase3 <- c("R6", "ggplot2", "dplyr", "tidyr",
                       "reshape2", "ReinforcementLearning")
  faltantes <- paquetes_clase3[!sapply(paquetes_clase3,
                                        function(p) requireNamespace(p, quietly = TRUE))]
  if (length(faltantes) > 0) {
    install.packages(faltantes, quiet = TRUE)
  }
  cat("Paquetes verificados.\n")
}

suppressMessages({
  library(R6)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
})

# Semilla global para reproducibilidad
SEMILLA_GLOBAL <- 42L
set.seed(SEMILLA_GLOBAL)

# Paleta institucional (consistente con la presentacion y el cuaderno Python)
COLOR_NAVY         <- "#1E3A5F"
COLOR_ORANGE       <- "#D86A2C"
COLOR_TEAL         <- "#2E8A99"
COLOR_LIGHT_TEAL   <- "#9FD3D8"
COLOR_LIGHT_ORANGE <- "#F2A06B"
COLOR_GRAY         <- "#4A4A4A"
COLOR_RED          <- "#B23A48"
COLOR_GREEN        <- "#2E7D5B"

# Tema grafico institucional
tema_austral <- theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(color = COLOR_NAVY, face = "bold", size = 12),
    axis.title = element_text(color = COLOR_NAVY),
    axis.text  = element_text(color = COLOR_GRAY),
    panel.grid.minor = element_blank(),
    legend.position = "right"
  )
theme_set(tema_austral)

cat("Entorno R configurado correctamente.\n")
cat("R version:", R.version.string, "\n")

## Bloque 1 — Entornos de trabajo

Construimos las contrapartes en R de los dos entornos del cuaderno Python. Usamos `R6` para clases con estado mutable, equivalente idiomático de las clases Python.

### 1.1. Random walk de 5 estados (Sutton & Barto, sección 6.2)

**Descripción.** Cadena lineal $T_{\text{izq}} - A - B - C - D - E - T_{\text{der}}$ con dos terminales en los extremos. Política uniforme entre izquierda/derecha. Recompensa $+1$ al absorber por la derecha, $0$ en otro caso. $\gamma = 1$.

**Valores verdaderos:** $V^\pi(A)=1/6,\ V^\pi(B)=2/6,\ V^\pi(C)=3/6,\ V^\pi(D)=4/6,\ V^\pi(E)=5/6$.

In [ ]:
RandomWalk <- R6Class("RandomWalk",
  public = list(
    n_estados = NULL,
    estado_inicial = NULL,
    V_verdadero = NULL,
    
    initialize = function() {
      self$n_estados <- 5L
      self$estado_inicial <- 3L  # C en indexacion 1-based de R: A=1, B=2, C=3, D=4, E=5
      self$V_verdadero <- c(1/6, 2/6, 3/6, 4/6, 5/6)
    },
    
    reset = function() {
      self$estado_inicial
    },
    
    paso = function(estado) {
      # Devuelve list(siguiente, recompensa, terminal)
      # Convencion: estado en {1,...,5} para internos; 0 y 6 son terminales
      accion <- sample(c(0, 1), 1)  # 0=izquierda, 1=derecha
      if (accion == 0) {
        siguiente <- estado - 1L
        recompensa <- 0
      } else {
        siguiente <- estado + 1L
        recompensa <- if (siguiente == 6L) 1 else 0
      }
      terminal <- (siguiente < 1L) || (siguiente > 5L)
      list(siguiente = siguiente, recompensa = recompensa, terminal = terminal)
    }
  )
)

rw <- RandomWalk$new()
cat("Random walk:", rw$n_estados, "estados internos.\n")
cat("Valores verdaderos:", round(rw$V_verdadero, 4), "\n")
cat("Estado inicial: C (indice", rw$estado_inicial, ")\n")

### 1.2. GridWorld estocástico

GridWorld 4x4 con dos muros internos y transiciones estocásticas: con probabilidad $1-p_{\text{slip}}$ se ejecuta la acción intencionada; con probabilidad $p_{\text{slip}}$ el agente se desvía ortogonalmente.

In [ ]:
GridWorldEstocastico <- R6Class("GridWorldEstocastico",
  public = list(
    n_filas = NULL,
    n_cols = NULL,
    muros = NULL,        # lista de vectores c(r, c)
    inicio = NULL,
    meta = NULL,
    recompensa_paso = NULL,
    recompensa_meta = NULL,
    p_slip = NULL,
    acciones = NULL,
    nombres_acciones = NULL,
    estados = NULL,
    idx_estado = NULL,
    n_estados = NULL,
    n_acciones = NULL,
    
    initialize = function(n_filas = 4, n_cols = 4,
                          muros = list(c(2, 1), c(2, 2)),
                          inicio = c(3, 0), meta = c(0, 3),
                          recompensa_paso = -0.04,
                          recompensa_meta = 1.0,
                          p_slip = 0.10) {
      self$n_filas <- n_filas
      self$n_cols <- n_cols
      self$muros <- muros
      self$inicio <- inicio
      self$meta <- meta
      self$recompensa_paso <- recompensa_paso
      self$recompensa_meta <- recompensa_meta
      self$p_slip <- p_slip
      
      # Acciones como desplazamientos (delta_fila, delta_col)
      # Orden: arriba, derecha, abajo, izquierda
      self$acciones <- list(c(-1, 0), c(0, 1), c(1, 0), c(0, -1))
      self$nombres_acciones <- c("arriba", "derecha", "abajo", "izquierda")
      
      # Construir lista de estados validos
      self$estados <- list()
      for (r in 0:(n_filas - 1)) {
        for (c in 0:(n_cols - 1)) {
          s <- c(r, c)
          if (!private$.es_muro(s)) {
            self$estados <- append(self$estados, list(s))
          }
        }
      }
      self$n_estados <- length(self$estados)
      self$n_acciones <- length(self$acciones)
      
      # Indice de estado: clave "r,c" -> entero
      claves <- sapply(self$estados, function(s) paste(s, collapse = ","))
      self$idx_estado <- setNames(seq_len(self$n_estados), claves)
    },
    
    indice = function(s) {
      clave <- paste(s, collapse = ",")
      self$idx_estado[[clave]]
    },
    
    aplicar_movimiento = function(estado, delta) {
      nr <- estado[1] + delta[1]
      nc <- estado[2] + delta[2]
      nuevo <- c(nr, nc)
      fuera <- (nr < 0 || nr >= self$n_filas || nc < 0 || nc >= self$n_cols)
      if (fuera || private$.es_muro(nuevo)) {
        return(estado)
      }
      nuevo
    },
    
    paso = function(estado, accion) {
      if (estado[1] == self$meta[1] && estado[2] == self$meta[2]) {
        return(list(siguiente = estado, recompensa = 0, terminal = TRUE))
      }
      
      # Estocasticidad
      if (runif(1) < self$p_slip) {
        ortogonales <- c((accion %% 4) + 1, ((accion - 2) %% 4) + 1)
        accion_real <- sample(ortogonales, 1)
      } else {
        accion_real <- accion
      }
      
      delta <- self$acciones[[accion_real]]
      siguiente <- self$aplicar_movimiento(estado, delta)
      
      if (siguiente[1] == self$meta[1] && siguiente[2] == self$meta[2]) {
        return(list(siguiente = siguiente,
                    recompensa = self$recompensa_meta,
                    terminal = TRUE))
      }
      list(siguiente = siguiente,
           recompensa = self$recompensa_paso,
           terminal = FALSE)
    },
    
    reset = function() {
      self$inicio
    }
  ),
  private = list(
    .es_muro = function(s) {
      for (m in self$muros) {
        if (s[1] == m[1] && s[2] == m[2]) return(TRUE)
      }
      FALSE
    }
  )
)

gw <- GridWorldEstocastico$new()
cat("GridWorld estocastico:", gw$n_estados, "estados,", gw$n_acciones, "acciones.\n")
cat("Probabilidad de deslizamiento:", gw$p_slip, "\n")
cat("Inicio:", gw$inicio, "  Meta:", gw$meta, "\n")

### 1.3. Generación de episodios

Funciones auxiliares que producen trayectorias. La convención de retorno es `list(estados = ..., recompensas = ...)` para facilitar el manejo vectorizado posterior.

In [ ]:
generar_episodio_random_walk <- function(env, max_pasos = 1000) {
  estados <- integer(0)
  recompensas <- numeric(0)
  estado <- env$reset()
  for (paso in 1:max_pasos) {
    res <- env$paso(estado)
    estados <- c(estados, estado)
    recompensas <- c(recompensas, res$recompensa)
    if (res$terminal) break
    estado <- res$siguiente
  }
  list(estados = estados, recompensas = recompensas)
}

generar_episodio_gridworld <- function(env, politica, max_pasos = 500) {
  # politica: matriz (n_estados, n_acciones)
  estados <- integer(0)
  recompensas <- numeric(0)
  estado <- env$reset()
  for (paso in 1:max_pasos) {
    idx <- env$indice(estado)
    accion <- sample(seq_len(env$n_acciones), 1, prob = politica[idx, ])
    res <- env$paso(estado, accion)
    estados <- c(estados, idx)
    recompensas <- c(recompensas, res$recompensa)
    if (res$terminal) break
    estado <- res$siguiente
  }
  list(estados = estados, recompensas = recompensas)
}

# Prueba: episodio en random walk
set.seed(SEMILLA_GLOBAL)
ep_demo <- generar_episodio_random_walk(rw)
cat("Longitud del episodio:", length(ep_demo$estados), "transiciones\n")
cat("Primeros estados visitados:", head(ep_demo$estados, 10), "\n")
cat("Recompensa terminal:", tail(ep_demo$recompensas, 1), "\n")

## Bloque 2 — Métodos Monte Carlo para la predicción

Implementación en R puro de las dos variantes clásicas: **first-visit** y **every-visit**. La actualización es incremental:

$$\widehat{V}(s) \leftarrow \widehat{V}(s) + \frac{1}{n+1}\left[G - \widehat{V}(s)\right].$$

In [ ]:
first_visit_mc_prediction <- function(env, generador_episodio, n_estados,
                                      n_episodios, gamma = 1.0,
                                      semilla = 0, V_verdadero = NULL,
                                      ...) {
  set.seed(semilla)
  V_hat <- rep(0.0, n_estados)
  conteo <- rep(0L, n_estados)
  rms_historial <- numeric(n_episodios)
  
  for (e in 1:n_episodios) {
    ep <- generador_episodio(env, ...)
    n_pasos <- length(ep$estados)
    
    # Calcular retornos hacia atras; first-visit: sobrescribir al iterar al reves
    G <- 0
    retornos_primera <- list()  # lista nombrada por estado
    for (t in n_pasos:1) {
      s <- ep$estados[t]
      r <- ep$recompensas[t]
      G <- gamma * G + r
      retornos_primera[[as.character(s)]] <- G
    }
    
    # Aplicar actualizacion incremental
    for (s_chr in names(retornos_primera)) {
      s <- as.integer(s_chr)
      G_primera <- retornos_primera[[s_chr]]
      conteo[s] <- conteo[s] + 1L
      V_hat[s] <- V_hat[s] + (G_primera - V_hat[s]) / conteo[s]
    }
    
    if (!is.null(V_verdadero)) {
      rms_historial[e] <- sqrt(mean((V_hat - V_verdadero)^2))
    }
  }
  
  list(V_hat = V_hat, rms = rms_historial)
}

In [ ]:
every_visit_mc_prediction <- function(env, generador_episodio, n_estados,
                                      n_episodios, gamma = 1.0,
                                      semilla = 0, V_verdadero = NULL,
                                      ...) {
  set.seed(semilla)
  V_hat <- rep(0.0, n_estados)
  conteo <- rep(0L, n_estados)
  rms_historial <- numeric(n_episodios)
  
  for (e in 1:n_episodios) {
    ep <- generador_episodio(env, ...)
    n_pasos <- length(ep$estados)
    
    # Calcular retornos de cada paso
    G <- 0
    retornos_paso <- numeric(n_pasos)
    estados_paso <- integer(n_pasos)
    for (t in n_pasos:1) {
      G <- gamma * G + ep$recompensas[t]
      retornos_paso[t] <- G
      estados_paso[t] <- ep$estados[t]
    }
    
    # Actualizacion por cada visita
    for (i in seq_along(estados_paso)) {
      s <- estados_paso[i]
      G_t <- retornos_paso[i]
      conteo[s] <- conteo[s] + 1L
      V_hat[s] <- V_hat[s] + (G_t - V_hat[s]) / conteo[s]
    }
    
    if (!is.null(V_verdadero)) {
      rms_historial[e] <- sqrt(mean((V_hat - V_verdadero)^2))
    }
  }
  
  list(V_hat = V_hat, rms = rms_historial)
}

### 2.2. Prueba rápida sobre el random walk

Ejecutamos 1000 episodios y comparamos con los valores verdaderos.

In [ ]:
res_fv <- first_visit_mc_prediction(rw, generar_episodio_random_walk,
                                    n_estados = rw$n_estados,
                                    n_episodios = 1000, gamma = 1.0,
                                    semilla = SEMILLA_GLOBAL,
                                    V_verdadero = rw$V_verdadero)
res_ev <- every_visit_mc_prediction(rw, generar_episodio_random_walk,
                                    n_estados = rw$n_estados,
                                    n_episodios = 1000, gamma = 1.0,
                                    semilla = SEMILLA_GLOBAL,
                                    V_verdadero = rw$V_verdadero)

tabla <- data.frame(
  Estado = c("A", "B", "C", "D", "E"),
  Verdadero = round(rw$V_verdadero, 4),
  FirstVisit = round(res_fv$V_hat, 4),
  EveryVisit = round(res_ev$V_hat, 4)
)
print(tabla)
cat("\nRMS first-visit:", sprintf("%.5f", tail(res_fv$rms, 1)), "\n")
cat("RMS every-visit:", sprintf("%.5f", tail(res_ev$rms, 1)), "\n")

## Bloque 3 — Diferencias Temporales TD(0)

Actualización canónica:

$$\widehat{V}(S_t) \leftarrow \widehat{V}(S_t) + \alpha\left[R_{t+1} + \gamma \widehat{V}(S_{t+1}) - \widehat{V}(S_t)\right].$$

In [ ]:
td0_prediction <- function(env, n_estados, n_episodios,
                           alpha = 0.1, gamma = 1.0,
                           semilla = 0, V_inicial = NULL,
                           V_verdadero = NULL,
                           politica = NULL,
                           modo = "random_walk") {
  set.seed(semilla)
  V_hat <- if (is.null(V_inicial)) rep(0.0, n_estados) else V_inicial
  rms_historial <- numeric(n_episodios)
  
  for (e in 1:n_episodios) {
    estado <- env$reset()
    
    repeat {
      if (modo == "random_walk") {
        idx_s <- estado
        res <- env$paso(estado)
        v_siguiente <- if (res$terminal) 0 else V_hat[res$siguiente]
      } else {
        idx_s <- env$indice(estado)
        accion <- sample(seq_len(env$n_acciones), 1, prob = politica[idx_s, ])
        res <- env$paso(estado, accion)
        v_siguiente <- if (res$terminal) 0 else V_hat[env$indice(res$siguiente)]
      }
      
      td_error <- res$recompensa + gamma * v_siguiente - V_hat[idx_s]
      V_hat[idx_s] <- V_hat[idx_s] + alpha * td_error
      
      if (res$terminal) break
      estado <- res$siguiente
    }
    
    if (!is.null(V_verdadero)) {
      rms_historial[e] <- sqrt(mean((V_hat - V_verdadero)^2))
    }
  }
  
  list(V_hat = V_hat, rms = rms_historial)
}

### 3.2. Prueba rápida sobre el random walk

Comparamos TD(0) con tres pasos de aprendizaje, inicializando $\widehat{V}_0(s) = 0.5$ siguiendo Sutton & Barto §6.2.

In [ ]:
V0 <- rep(0.5, rw$n_estados)

resultados_td <- list()
for (alpha in c(0.05, 0.10, 0.15)) {
  res <- td0_prediction(rw, rw$n_estados, n_episodios = 1000,
                        alpha = alpha, gamma = 1.0,
                        semilla = SEMILLA_GLOBAL,
                        V_inicial = V0,
                        V_verdadero = rw$V_verdadero,
                        modo = "random_walk")
  resultados_td[[as.character(alpha)]] <- res$V_hat
}

tabla_td <- data.frame(
  Estado = c("A", "B", "C", "D", "E"),
  Verdadero = round(rw$V_verdadero, 4),
  alpha_005 = round(resultados_td[["0.05"]], 4),
  alpha_010 = round(resultados_td[["0.1"]], 4),
  alpha_015 = round(resultados_td[["0.15"]], 4)
)
names(tabla_td) <- c("Estado", "Verdadero", "alpha=0.05", "alpha=0.10", "alpha=0.15")
print(tabla_td)

## Bloque 4 — Réplica del experimento canónico (Sutton & Barto, figura 6.2)

Replicamos el experimento más conocido del capítulo 6: comparación del error RMS de Monte Carlo (varios $\alpha$) y TD(0) (varios $\alpha$) sobre el random walk, promediando 100 corridas independientes.

In [ ]:
# Variante MC con paso constante (necesaria para reproducir la figura)
mc_paso_constante <- function(env, n_estados, n_episodios, alpha,
                              gamma = 1.0, semilla = 0,
                              V_inicial = NULL, V_verdadero = NULL) {
  set.seed(semilla)
  V_hat <- if (is.null(V_inicial)) rep(0.0, n_estados) else V_inicial
  rms_historial <- numeric(n_episodios)
  
  for (e in 1:n_episodios) {
    ep <- generar_episodio_random_walk(env)
    n_pasos <- length(ep$estados)
    G <- 0
    primeros <- list()
    for (t in n_pasos:1) {
      G <- gamma * G + ep$recompensas[t]
      primeros[[as.character(ep$estados[t])]] <- G
    }
    for (s_chr in names(primeros)) {
      s <- as.integer(s_chr)
      V_hat[s] <- V_hat[s] + alpha * (primeros[[s_chr]] - V_hat[s])
    }
    if (!is.null(V_verdadero)) {
      rms_historial[e] <- sqrt(mean((V_hat - V_verdadero)^2))
    }
  }
  
  list(V_hat = V_hat, rms = rms_historial)
}

N_REPLICAS <- 100L
N_EPISODIOS <- 100L

mc_alphas <- c(0.01, 0.02, 0.03, 0.04)
td_alphas <- c(0.05, 0.10, 0.15)

resultados_mc <- list()
for (alpha in mc_alphas) {
  cat("MC con alpha =", alpha, "...")
  rms_mat <- matrix(0, nrow = N_REPLICAS, ncol = N_EPISODIOS)
  for (i in 1:N_REPLICAS) {
    res <- mc_paso_constante(rw, rw$n_estados, N_EPISODIOS, alpha,
                              gamma = 1.0,
                              semilla = SEMILLA_GLOBAL + i,
                              V_inicial = V0,
                              V_verdadero = rw$V_verdadero)
    rms_mat[i, ] <- res$rms
  }
  resultados_mc[[as.character(alpha)]] <- rms_mat
  cat(" OK\n")
}

resultados_td_rep <- list()
for (alpha in td_alphas) {
  cat("TD(0) con alpha =", alpha, "...")
  rms_mat <- matrix(0, nrow = N_REPLICAS, ncol = N_EPISODIOS)
  for (i in 1:N_REPLICAS) {
    res <- td0_prediction(rw, rw$n_estados, N_EPISODIOS,
                          alpha = alpha, gamma = 1.0,
                          semilla = SEMILLA_GLOBAL + i,
                          V_inicial = V0,
                          V_verdadero = rw$V_verdadero,
                          modo = "random_walk")
    rms_mat[i, ] <- res$rms
  }
  resultados_td_rep[[as.character(alpha)]] <- rms_mat
  cat(" OK\n")
}

cat("\nTodas las replicas completadas.\n")

In [ ]:
# Construir data frame largo para ggplot
df_curvas <- data.frame()
for (alpha in td_alphas) {
  media <- colMeans(resultados_td_rep[[as.character(alpha)]])
  df_curvas <- rbind(df_curvas, data.frame(
    Episodio = 1:N_EPISODIOS, RMS = media,
    Metodo = paste0("TD(0), alpha=", alpha),
    Tipo = "TD(0)",
    Alpha = alpha
  ))
}
for (alpha in mc_alphas) {
  media <- colMeans(resultados_mc[[as.character(alpha)]])
  df_curvas <- rbind(df_curvas, data.frame(
    Episodio = 1:N_EPISODIOS, RMS = media,
    Metodo = paste0("MC, alpha=", alpha),
    Tipo = "MC",
    Alpha = alpha
  ))
}

paleta <- c(
  "TD(0), alpha=0.05" = "#0F2A4F",
  "TD(0), alpha=0.1"  = COLOR_NAVY,
  "TD(0), alpha=0.15" = "#3A5F8F",
  "MC, alpha=0.01"    = "#7A3812",
  "MC, alpha=0.02"    = "#A14A1A",
  "MC, alpha=0.03"    = COLOR_ORANGE,
  "MC, alpha=0.04"    = "#F4854B"
)

p_replica <- ggplot(df_curvas, aes(x = Episodio, y = RMS, color = Metodo,
                                    linetype = Tipo)) +
  geom_line(linewidth = 0.9) +
  scale_color_manual(values = paleta) +
  scale_linetype_manual(values = c("TD(0)" = "solid", "MC" = "dashed")) +
  labs(title = "Convergencia de TD(0) vs Monte Carlo en el random walk",
       subtitle = "Replica de Sutton & Barto, figura 6.2",
       x = "Episodios",
       y = "Error RMS sobre estados (promedio 100 replicas)") +
  guides(linetype = "none") +
  ylim(0, 0.30)

print(p_replica)

cat("\nLectura: TD(0) con pasos moderados converge mas rapido.\n")
cat("MC con alpha pequenos es mas estable a largo plazo pero mas lento al inicio.\n")

## Bloque 5 — Análisis de sensibilidad al paso $\alpha$

Barrido sobre $\alpha \in \{0.01, 0.05, 0.10, 0.15, 0.20, 0.30\}$ con 100 réplicas independientes.

In [ ]:
alphas_barrido <- c(0.01, 0.05, 0.10, 0.15, 0.20, 0.30)
N_REP_BARRIDO <- 100L
N_EP_BARRIDO <- 100L

rms_finales_td <- list()
rms_finales_mc <- list()

for (alpha in alphas_barrido) {
  finales_td <- numeric(N_REP_BARRIDO)
  finales_mc <- numeric(N_REP_BARRIDO)
  for (i in 1:N_REP_BARRIDO) {
    res_td <- td0_prediction(rw, rw$n_estados, N_EP_BARRIDO,
                              alpha = alpha, gamma = 1.0,
                              semilla = SEMILLA_GLOBAL + i,
                              V_inicial = V0,
                              V_verdadero = rw$V_verdadero,
                              modo = "random_walk")
    finales_td[i] <- tail(res_td$rms, 1)
    
    res_mc <- mc_paso_constante(rw, rw$n_estados, N_EP_BARRIDO, alpha,
                                 gamma = 1.0,
                                 semilla = SEMILLA_GLOBAL + i,
                                 V_inicial = V0,
                                 V_verdadero = rw$V_verdadero)
    finales_mc[i] <- tail(res_mc$rms, 1)
  }
  rms_finales_td[[as.character(alpha)]] <- finales_td
  rms_finales_mc[[as.character(alpha)]] <- finales_mc
}

cat("Barrido sobre alpha completado.\n")

In [ ]:
df_barrido <- data.frame()
for (alpha in alphas_barrido) {
  vals_td <- rms_finales_td[[as.character(alpha)]]
  vals_mc <- rms_finales_mc[[as.character(alpha)]]
  df_barrido <- rbind(df_barrido,
    data.frame(Alpha = alpha, Media = mean(vals_td),
               EE = sd(vals_td) / sqrt(length(vals_td)),
               Metodo = "TD(0)"),
    data.frame(Alpha = alpha, Media = mean(vals_mc),
               EE = sd(vals_mc) / sqrt(length(vals_mc)),
               Metodo = "Monte Carlo")
  )
}

p_barrido <- ggplot(df_barrido, aes(x = Alpha, y = Media, color = Metodo)) +
  geom_line(linewidth = 1, aes(linetype = Metodo)) +
  geom_point(size = 3) +
  geom_errorbar(aes(ymin = Media - 1.96 * EE, ymax = Media + 1.96 * EE),
                width = 0.05, linewidth = 0.6) +
  scale_color_manual(values = c("TD(0)" = COLOR_NAVY,
                                 "Monte Carlo" = COLOR_ORANGE)) +
  scale_x_log10(breaks = alphas_barrido) +
  labs(title = "Sensibilidad del error final al paso alpha",
       x = "Paso de aprendizaje alpha (escala log)",
       y = paste0("RMS final tras ", N_EP_BARRIDO, " episodios (IC 95%)"))

print(p_barrido)

medias_td_vec <- sapply(alphas_barrido, function(a) mean(rms_finales_td[[as.character(a)]]))
medias_mc_vec <- sapply(alphas_barrido, function(a) mean(rms_finales_mc[[as.character(a)]]))
alpha_opt_td <- alphas_barrido[which.min(medias_td_vec)]
alpha_opt_mc <- alphas_barrido[which.min(medias_mc_vec)]
cat("\nalpha optimo TD(0):", alpha_opt_td, "  RMS =", round(min(medias_td_vec), 4), "\n")
cat("alpha optimo MC:   ", alpha_opt_mc, "  RMS =", round(min(medias_mc_vec), 4), "\n")

## Bloque 6 — Análisis de varianza con N=500 réplicas

Escalamos a $N=500$ réplicas para construir bandas de confianza tipo bootstrap-percentil y compararlas con el IC gaussiano al 95%.

In [ ]:
N_GRANDE <- 500L
N_EP_GRANDE <- 50L

# TD(0) con alpha = 0.10
matriz_td_grande <- matrix(0, nrow = N_GRANDE, ncol = N_EP_GRANDE)
for (i in 1:N_GRANDE) {
  res <- td0_prediction(rw, rw$n_estados, N_EP_GRANDE,
                        alpha = 0.10, gamma = 1.0,
                        semilla = SEMILLA_GLOBAL + i,
                        V_inicial = V0,
                        V_verdadero = rw$V_verdadero,
                        modo = "random_walk")
  matriz_td_grande[i, ] <- res$rms
}

# MC con alpha = 0.02
matriz_mc_grande <- matrix(0, nrow = N_GRANDE, ncol = N_EP_GRANDE)
for (i in 1:N_GRANDE) {
  res <- mc_paso_constante(rw, rw$n_estados, N_EP_GRANDE, 0.02,
                            gamma = 1.0,
                            semilla = SEMILLA_GLOBAL + i,
                            V_inicial = V0,
                            V_verdadero = rw$V_verdadero)
  matriz_mc_grande[i, ] <- res$rms
}

cat("Replicas N=500 completadas.\n")

In [ ]:
# Construir data frame con bandas
construir_df_bandas <- function(matriz, nombre) {
  data.frame(
    Episodio = 1:ncol(matriz),
    Media = colMeans(matriz),
    Q025 = apply(matriz, 2, quantile, probs = 0.025),
    Q975 = apply(matriz, 2, quantile, probs = 0.975),
    SD = apply(matriz, 2, sd),
    N = nrow(matriz),
    Metodo = nombre
  )
}

df_td <- construir_df_bandas(matriz_td_grande, "TD(0), alpha=0.10")
df_mc <- construir_df_bandas(matriz_mc_grande, "MC, alpha=0.02")

# Combinar y agregar IC gaussiano
df_bandas <- rbind(df_td, df_mc)
df_bandas$IC_inf <- df_bandas$Media - 1.96 * df_bandas$SD / sqrt(df_bandas$N)
df_bandas$IC_sup <- df_bandas$Media + 1.96 * df_bandas$SD / sqrt(df_bandas$N)
df_bandas$Metodo <- factor(df_bandas$Metodo,
                            levels = c("TD(0), alpha=0.10", "MC, alpha=0.02"))

# Paleta facetada
paleta_facet <- c("TD(0), alpha=0.10" = COLOR_NAVY,
                  "MC, alpha=0.02"    = COLOR_ORANGE)

p_bandas <- ggplot(df_bandas, aes(x = Episodio)) +
  geom_ribbon(aes(ymin = Q025, ymax = Q975, fill = Metodo), alpha = 0.2) +
  geom_ribbon(aes(ymin = IC_inf, ymax = IC_sup, fill = Metodo), alpha = 0.5) +
  geom_line(aes(y = Media, color = Metodo), linewidth = 1.1) +
  scale_color_manual(values = paleta_facet) +
  scale_fill_manual(values = paleta_facet) +
  facet_wrap(~ Metodo, ncol = 2) +
  labs(title = "Bandas de confianza al 95% (N=500 replicas)",
       subtitle = "Sombreado claro: percentiles. Sombreado oscuro: IC gaussiano de la media.",
       x = "Episodios", y = "Error RMS") +
  ylim(0, 0.30) +
  guides(color = "none", fill = "none")

print(p_bandas)

cat("\nDesvio estandar final TD(0):", round(tail(df_td$SD, 1), 4), "\n")
cat("Desvio estandar final MC:   ", round(tail(df_mc$SD, 1), 4), "\n")

## Bloque 7 — Aplicación al GridWorld estocástico

Aplicamos los métodos al GridWorld con política uniforme y comparamos con la solución exacta obtenida por inversión matricial.

### 7.1. Cálculo del valor verdadero por inversión exacta

In [ ]:
calcular_dinamica_inducida_gridworld <- function(env, politica) {
  n <- env$n_estados
  P_pi <- matrix(0, nrow = n, ncol = n)
  R_pi <- rep(0, n)
  
  for (s in env$estados) {
    i <- env$indice(s)
    if (s[1] == env$meta[1] && s[2] == env$meta[2]) {
      P_pi[i, i] <- 1
      next
    }
    for (a in 1:env$n_acciones) {
      pi_sa <- politica[i, a]
      # Tres destinos: accion intencionada y dos ortogonales
      a_orto1 <- (a %% 4) + 1
      a_orto2 <- ((a - 2) %% 4) + 1
      destinos <- list(
        list(prob = 1 - env$p_slip, accion = a),
        list(prob = env$p_slip / 2, accion = a_orto1),
        list(prob = env$p_slip / 2, accion = a_orto2)
      )
      for (d in destinos) {
        delta <- env$acciones[[d$accion]]
        s_prima <- env$aplicar_movimiento(s, delta)
        j <- env$indice(s_prima)
        es_meta <- (s_prima[1] == env$meta[1] && s_prima[2] == env$meta[2])
        r <- if (es_meta) env$recompensa_meta else env$recompensa_paso
        P_pi[i, j] <- P_pi[i, j] + pi_sa * d$prob
        R_pi[i] <- R_pi[i] + pi_sa * d$prob * r
      }
    }
  }
  list(P = P_pi, R = R_pi)
}

# Politica uniforme
politica_uniforme <- matrix(1 / gw$n_acciones,
                             nrow = gw$n_estados, ncol = gw$n_acciones)

gamma_gw <- 0.95
din <- calcular_dinamica_inducida_gridworld(gw, politica_uniforme)
V_verdadero_gw <- solve(diag(gw$n_estados) - gamma_gw * din$P, din$R)

cat("Valor exacto calculado para los", gw$n_estados, "estados del GridWorld.\n")
cat("V*(inicio) =", round(V_verdadero_gw[gw$indice(gw$inicio)], 4), "\n")
cat("V*(meta)   =", round(V_verdadero_gw[gw$indice(gw$meta)], 4), "\n")

### 7.2. Estimación por MC y TD(0)

In [ ]:
# Wrapper para que generador_episodio reciba politica
gen_gw_wrap <- function(env, ...) {
  generar_episodio_gridworld(env, politica_uniforme, ...)
}

N_EP_GW <- 5000L

res_mc_gw <- first_visit_mc_prediction(gw, gen_gw_wrap,
                                       n_estados = gw$n_estados,
                                       n_episodios = N_EP_GW,
                                       gamma = gamma_gw,
                                       semilla = SEMILLA_GLOBAL)
V_mc_gw <- res_mc_gw$V_hat

res_td_gw <- td0_prediction(gw, gw$n_estados, N_EP_GW,
                            alpha = 0.1, gamma = gamma_gw,
                            semilla = SEMILLA_GLOBAL,
                            politica = politica_uniforme,
                            modo = "gridworld")
V_td_gw <- res_td_gw$V_hat

cat("GridWorld - tras", N_EP_GW, "episodios:\n")
cat("V_MC(inicio)        =", round(V_mc_gw[gw$indice(gw$inicio)], 4), "\n")
cat("V_TD(inicio)        =", round(V_td_gw[gw$indice(gw$inicio)], 4), "\n")
cat("V_verdadero(inicio) =", round(V_verdadero_gw[gw$indice(gw$inicio)], 4), "\n")

In [ ]:
# Construir mapas de valores en formato largo
construir_df_grilla <- function(env, valores, etiqueta) {
  df <- data.frame()
  for (s in env$estados) {
    df <- rbind(df, data.frame(
      Fila = s[1], Columna = s[2],
      Valor = valores[env$indice(s)],
      Etiqueta = etiqueta
    ))
  }
  # Agregar muros como NA para que aparezcan grises
  for (m in env$muros) {
    df <- rbind(df, data.frame(
      Fila = m[1], Columna = m[2],
      Valor = NA, Etiqueta = etiqueta
    ))
  }
  df
}

df_verd <- construir_df_grilla(gw, V_verdadero_gw, "Verdadero (inversion exacta)")
df_mc_g <- construir_df_grilla(gw, V_mc_gw, paste0("Monte Carlo (E=", N_EP_GW, ")"))
df_td_g <- construir_df_grilla(gw, V_td_gw, paste0("TD(0) alpha=0.10 (E=", N_EP_GW, ")"))
df_grillas <- rbind(df_verd, df_mc_g, df_td_g)
df_grillas$Etiqueta <- factor(df_grillas$Etiqueta,
  levels = unique(df_grillas$Etiqueta))

vmin <- min(df_grillas$Valor, na.rm = TRUE)
vmax <- max(df_grillas$Valor, na.rm = TRUE)

p_grilla <- ggplot(df_grillas, aes(x = Columna, y = -Fila, fill = Valor)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = ifelse(is.na(Valor), "", sprintf("%.2f", Valor))),
            color = "black", size = 3) +
  scale_fill_gradient2(low = COLOR_RED, mid = "#FFEFC4",
                       high = COLOR_GREEN, midpoint = 0,
                       limits = c(vmin, vmax),
                       na.value = COLOR_GRAY) +
  facet_wrap(~ Etiqueta, ncol = 3) +
  coord_equal() +
  labs(title = "Comparacion de estimadores en GridWorld estocastico",
       subtitle = "politica uniforme",
       x = NULL, y = NULL, fill = expression(V^pi(s))) +
  theme(axis.text = element_blank(),
        axis.ticks = element_blank(),
        panel.grid = element_blank())

print(p_grilla)

In [ ]:
# Mapas de error absoluto
err_mc <- abs(V_mc_gw - V_verdadero_gw)
err_td <- abs(V_td_gw - V_verdadero_gw)
rms_mc <- sqrt(mean(err_mc^2))
rms_td <- sqrt(mean(err_td^2))

df_err_mc <- construir_df_grilla(gw, err_mc,
  sprintf("Error |V_MC - V*| (RMS=%.4f)", rms_mc))
df_err_td <- construir_df_grilla(gw, err_td,
  sprintf("Error |V_TD - V*| (RMS=%.4f)", rms_td))
df_errores <- rbind(df_err_mc, df_err_td)
df_errores$Etiqueta <- factor(df_errores$Etiqueta,
  levels = unique(df_errores$Etiqueta))

vmax_err <- max(c(err_mc, err_td), na.rm = TRUE)

p_errores <- ggplot(df_errores, aes(x = Columna, y = -Fila, fill = Valor)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = ifelse(is.na(Valor), "", sprintf("%.2f", Valor))),
            color = "black", size = 3) +
  scale_fill_gradient(low = "white", high = COLOR_RED,
                      limits = c(0, vmax_err),
                      na.value = COLOR_GRAY) +
  facet_wrap(~ Etiqueta, ncol = 2) +
  coord_equal() +
  labs(title = "Mapas de error: donde cada metodo se aparta del valor verdadero",
       x = NULL, y = NULL, fill = "Error absoluto") +
  theme(axis.text = element_blank(),
        axis.ticks = element_blank(),
        panel.grid = element_blank())

print(p_errores)

cat("\nError RMS Monte Carlo:", round(rms_mc, 4), "\n")
cat("Error RMS TD(0):      ", round(rms_td, 4), "\n")

## Sección complementaria — Uso del paquete `ReinforcementLearning`

El paquete [`ReinforcementLearning`](https://cran.r-project.org/package=ReinforcementLearning) ofrece una implementación tabular ya disponible en CRAN. Es ilustrativo verificar nuestra implementación contrastándola con esa herramienta canónica. La API del paquete espera un *data frame* con cuatro columnas: `State`, `Action`, `Reward`, `NextState`.

> **Nota.** Esta sección se ejecuta solo si el paquete está instalado. En el entorno local del docente, ya forma parte de las dependencias del entorno R 4.5.2.

In [ ]:
if (requireNamespace("ReinforcementLearning", quietly = TRUE)) {
  library(ReinforcementLearning)
  
  # Generar un data set de transiciones desde GridWorld
  set.seed(SEMILLA_GLOBAL)
  N_TRANSICIONES <- 10000L
  
  transiciones <- data.frame(
    State = character(N_TRANSICIONES),
    Action = character(N_TRANSICIONES),
    Reward = numeric(N_TRANSICIONES),
    NextState = character(N_TRANSICIONES),
    stringsAsFactors = FALSE
  )
  
  idx <- 1
  estado <- gw$reset()
  for (i in 1:N_TRANSICIONES) {
    s_lab <- paste0("s_", gw$indice(estado))
    a_idx <- sample(seq_len(gw$n_acciones), 1, prob = politica_uniforme[gw$indice(estado), ])
    a_lab <- gw$nombres_acciones[a_idx]
    res <- gw$paso(estado, a_idx)
    ns_lab <- paste0("s_", gw$indice(res$siguiente))
    transiciones[i, ] <- list(s_lab, a_lab, res$recompensa, ns_lab)
    if (res$terminal) {
      estado <- gw$reset()
    } else {
      estado <- res$siguiente
    }
  }
  
  # Entrenar usando el paquete (algoritmo Q-Learning interno por defecto)
  control <- list(alpha = 0.1, gamma = gamma_gw, epsilon = 0.1)
  modelo_rl <- ReinforcementLearning(transiciones,
                                      s = "State", a = "Action",
                                      r = "Reward", s_new = "NextState",
                                      control = control)
  
  cat("Modelo ReinforcementLearning entrenado.\n")
  cat("\nValor estimado V(s) marginalizando sobre acciones (politica greedy):\n")
  # El paquete almacena Q en modelo_rl$Q
  Q_estimado <- modelo_rl$Q
  V_pkg <- apply(Q_estimado, 1, max)
  print(round(V_pkg, 3))
  
} else {
  cat("Paquete ReinforcementLearning no disponible.\n")
  cat("Instalelo con: install.packages('ReinforcementLearning')\n")
}

## Ejercicios propuestos

Los ejercicios mantienen exacta correspondencia con los del cuaderno Python. El plazo de entrega del componente evaluado de la Unidad 2 es el **25 de mayo, 23:59 hs**.

### Ejercicios obligatorios (formales)

1. **Ejercicio teórico 1.** Demuestre que el estimador first-visit MC del valor de un estado $s$ es insesgado. Identifique con precisión la fuente de aleatoriedad implicada en la esperanza.

2. **Ejercicio teórico 2.** Pruebe que la actualización TD(0) con pasos $\alpha_n(s)$ que verifican Robbins-Monro converge en esperanza al punto fijo $V^\pi$. *(Sugerencia: trate la actualización como un esquema estocástico sobre el operador $\mathcal{T}^\pi$ y emplee el lema de Robbins-Siegmund.)*

3. **Ejercicio computacional 1.** Reproduzca la figura 6.2 de Sutton & Barto (Bloque 4 de este cuaderno) variando el número de réplicas a $N = 200$ y agregue una grilla más densa de $\alpha$. Discuta si el cruce de las curvas MC y TD se mantiene.

4. **Ejercicio computacional 2.** Compare empíricamente la sensibilidad de MC y TD(0) al paso $\alpha$ sobre GridWorld estocástico. ¿Existe un $\alpha^*$ que minimice el error RMS asintótico? Documente sus hallazgos con figuras y comentarios.

### Ejercicios opcionales (de profundización)

5. **Ejercicio integrador.** Implemente un *n-step TD* (versión intermedia entre TD(0) y MC):

   $$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{n-1} R_{t+n} + \gamma^n \widehat{V}(S_{t+n}).$$

   Compare el error final como función de $n \in \{1, 2, 4, 8, 16\}$ sobre el random walk.

6. **Ejercicio teórico avanzado.** Discuta por qué en el régimen *batch* Monte Carlo converge al estimador de mínimos cuadrados sobre las muestras observadas, mientras que TD(0) converge al estimador de máxima verosimilitud del MDP empírico.

7. **Ejercicio R-específico.** Compare la velocidad de ejecución entre la implementación nativa en R puro de este cuaderno y la del paquete `ReinforcementLearning` para entrenar 5000 transiciones. Use `system.time` para cronometrar. Discuta los compromisos.

---

### Referencias del cuaderno

* Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2.ª ed.). MIT Press. Capítulos 5 y 6.
* Zhao, S. (2024). *Mathematical Foundations of Reinforcement Learning*. Springer. Capítulos 5 y 7.
* Bertsekas, D. P., & Tsitsiklis, J. N. (1996). *Neuro-Dynamic Programming*. Athena Scientific.
* Proellochs, N., & Feuerriegel, S. (2017). *ReinforcementLearning: Model-Free Reinforcement Learning*. R package version 1.0.5.

---

*Cuaderno preparado para la Clase 3 del curso «Aprendizaje por Refuerzo: Fundamentos y Aplicaciones», Universidad Austral, mayo de 2026.*